In [11]:
# Load raw counts object
adata1_raw = sc.read_h5ad(RAW_DIR / "GSE114725_raw.h5ad")
adata2_raw = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")

# Subset raw to only the cells that survived QC
# using the cell barcodes from our annotated objects
adata1_raw_qc = adata1_raw[adata1.obs_names].copy()
adata2_raw_qc = adata2_raw[adata2.obs_names].copy()

# Transfer cell type and condition labels
adata1_raw_qc.obs["cell_type"] = adata1.obs["cell_type"].values
adata1_raw_qc.obs["tissue"] = adata1.obs["tissue"].values
adata1_raw_qc.obs["patient"] = adata1.obs["patient"].values

adata2_raw_qc.obs["cell_type"] = adata2.obs["cell_type"].values
adata2_raw_qc.obs["subtype"] = adata2.obs["subtype"].values
adata2_raw_qc.obs["orig.ident"] = adata2.obs["orig.ident"].values

# Verify raw counts
print("GSE114725 raw max:", adata1_raw_qc.X.max())
print("GSE176078 raw max:", adata2_raw_qc.X.max())
print("GSE114725 shape:", adata1_raw_qc.shape)
print("GSE176078 shape:", adata2_raw_qc.shape)

GSE114725 raw max: 1236.0
GSE176078 raw max: 29831.0
GSE114725 shape: (44662, 14875)
GSE176078 shape: (91425, 29733)


In [13]:
# Cell 2 — Load only obs metadata to save memory
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata1.obs["cell_type"] = adata1.obs["leiden_0.8"].map(cluster_labels_1)

# Extract just the obs metadata we need from adata2 without loading X
import h5py
import pandas as pd

with h5py.File(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad", "r") as f:
    obs2 = pd.DataFrame({
        "leiden_0.6": f["obs"]["leiden_0.6"]["codes"][:].astype(str),
        "subtype": f["obs"]["subtype"]["codes"][:].astype(str),
        "orig.ident": f["obs"]["orig.ident"]["codes"][:].astype(str),
    })

print("adata1 loaded:", adata1.n_obs, "cells")
print("adata2 obs extracted:", obs2.shape)
print(obs2.head())

adata1 loaded: 44662 cells
adata2 obs extracted: (91425, 3)
  leiden_0.6 subtype orig.ident
0          0       1          0
1          0       1          0
2          0       1          0
3          0       1          0
4          0       1          0


In [15]:
# Cell 2 — Load adata1 and extract adata2 obs only
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata1.obs["cell_type"] = adata1.obs["leiden_0.8"].map(cluster_labels_1)
gc.collect()

# For adata2 — read only the obs dataframe using pandas/h5py
import h5py

with h5py.File(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad", "r") as f:
    # Get category codes and categories for leiden_0.6
    leiden_codes = f["obs"]["leiden_0.6"]["codes"][:]
    leiden_cats = f["obs"]["leiden_0.6"]["categories"][:].astype(str)
    
    # Get subtype
    subtype_codes = f["obs"]["subtype"]["codes"][:]
    subtype_cats = f["obs"]["subtype"]["categories"][:].astype(str)
    
    # Get orig.ident
    ident_codes = f["obs"]["orig.ident"]["codes"][:]
    ident_cats = f["obs"]["orig.ident"]["categories"][:].astype(str)
    
    # Get cell barcodes (index)
    barcodes = f["obs"]["_index"][:].astype(str)

obs2 = pd.DataFrame({
    "leiden_0.6": leiden_cats[leiden_codes],
    "subtype": subtype_cats[subtype_codes],
    "orig.ident": ident_cats[ident_codes],
}, index=barcodes)

obs2["cell_type"] = obs2["leiden_0.6"].map(cluster_labels_2)

gc.collect()

print("adata1:", adata1.n_obs, "cells")
print("adata2 obs:", obs2.shape)
print("adata2 cell types:", obs2["cell_type"].value_counts().to_dict())

MemoryError: Unable to allocate 681. MiB for an array with shape (44662, 2000) and data type float64

In [10]:
# Cell 1 — Imports and paths
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_labels_1 = {
    "0": "T cells (resting)", "1": "T cells (naive/memory)",
    "2": "NK/Cytotoxic T cells", "3": "Activated T cells",
    "4": "Macrophages", "5": "Monocytes/DC"
}

cluster_labels_2 = {
    "0": "Endothelial cells", "1": "Endothelial cells",
    "2": "CAFs", "3": "PVL", "4": "Basal epithelial",
    "5": "B cells", "6": "Cycling cells", "7": "Plasma cells",
    "8": "Cycling epithelial", "9": "CD8 T cells",
    "10": "NK cells", "11": "T cells", "12": "Naive/memory T cells",
    "13": "Luminal epithelial", "14": "Macrophages",
    "15": "Monocytes/DC", "16": "Cycling myeloid", "17": "pDC",
    "18": "Luminal epithelial", "19": "Luminal epithelial",
    "20": "Epithelial", "21": "Epithelial",
    "22": "Luminal epithelial", "23": "Luminal epithelial",
    "24": "Luminal epithelial", "25": "Luminal epithelial"
}

IMMUNE_TYPES = [
    "T cells", "CD8 T cells", "NK cells", "Naive/memory T cells",
    "B cells", "Plasma cells", "Macrophages", "Monocytes/DC", "pDC"
]

print("Imports done")

Imports done


In [2]:
# ----------------------------
# Cell 2 — Load data
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata1.obs["cell_type"] = adata1.obs["leiden_0.8"].map(cluster_labels_1)

adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad")
adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

print("GSE114725:", adata1.n_obs, "cells")
print("  Cell types:", adata1.obs["cell_type"].value_counts().to_dict())
print("\nGSE176078:", adata2.n_obs, "cells")
print("  Tissue types:", adata1.obs["tissue"].unique().tolist())
print("  Subtypes:", adata2.obs["subtype"].unique().tolist())

GSE114725: 44662 cells
  Cell types: {'T cells (resting)': 11734, 'Macrophages': 8691, 'NK/Cytotoxic T cells': 8676, 'Activated T cells': 7141, 'T cells (naive/memory)': 4341, 'Monocytes/DC': 4079}

GSE176078: 91425 cells
  Tissue types: ['TUMOR', 'NORMAL', 'LYMPHNODE', 'BLOOD']
  Subtypes: ['HER2+', 'TNBC', 'ER+']


In [ ]:
# ----------------------------
# Cell 3 — DE analysis GSE114725
# Tumour vs Blood within each cell type
# Using Scanpy Wilcoxon (awaiting Adrien confirmation on pseudobulk)
# ----------------------------

# Subset to TUMOR and BLOOD only for clean comparison
adata1_tb = adata1[
    adata1.obs["tissue"].isin(["TUMOR", "BLOOD"])
].copy()

print(f"TUMOR + BLOOD cells: {adata1_tb.n_obs}")
print(adata1_tb.obs["tissue"].value_counts())

# Create combined label for grouping
adata1_tb.obs["celltype_tissue"] = (
    adata1_tb.obs["cell_type"].astype(str) + "_" +
    adata1_tb.obs["tissue"].astype(str)
)

print("\nCell type x tissue groups:")
print(adata1_tb.obs["celltype_tissue"].value_counts())

In [ ]:
# ----------------------------
# Cell 4 — Run DE per cell type (TUMOR vs BLOOD)
# ----------------------------
cell_types_de = [
    "T cells (resting)",
    "NK/Cytotoxic T cells",
    "Activated T cells",
    "Macrophages"
]

de_results = {}

for ct in cell_types_de:
    print(f"\nRunning DE for {ct}...")
    
    # Subset to this cell type
    adata_ct = adata1_tb[adata1_tb.obs["cell_type"] == ct].copy()
    
    # Check we have enough cells in both conditions
    counts = adata_ct.obs["tissue"].value_counts()
    print(f"  TUMOR: {counts.get('TUMOR', 0)}, BLOOD: {counts.get('BLOOD', 0)}")
    
    if counts.get("TUMOR", 0) < 20 or counts.get("BLOOD", 0) < 20:
        print(f"  Skipping — not enough cells")
        continue
    
    # Run Wilcoxon DE
    sc.tl.rank_genes_groups(
        adata_ct,
        groupby="tissue",
        groups=["TUMOR"],
        reference="BLOOD",
        method="wilcoxon",
        key_added="de_tumor_vs_blood"
    )
    
    # Extract results
    de_df = sc.get.rank_genes_groups_df(
        adata_ct,
        group="TUMOR",
        key="de_tumor_vs_blood"
    )
    
    # Multiple testing correction already applied (BH)
    # Filter significant genes
    de_sig = de_df[
        (de_df["pvals_adj"] < 0.05) &
        (abs(de_df["logfoldchanges"]) > 0.5)
    ].copy()
    
    de_results[ct] = de_df
    
    print(f"  Significant DEGs: {len(de_sig)}")
    print(f"  Top upregulated in TUMOR:")
    print(de_sig[de_sig["logfoldchanges"] > 0].head(5)[["names", "logfoldchanges", "pvals_adj"]].to_string())
    print(f"  Top downregulated in TUMOR:")
    print(de_sig[de_sig["logfoldchanges"] < 0].head(5)[["names", "logfoldchanges", "pvals_adj"]].to_string())
    
    # Save
    de_df.to_csv(
        RESULTS_DIR / f"GSE114725_DE_{ct.replace('/', '_').replace(' ', '_')}_tumor_vs_blood.csv",
        index=False
    )

print("\nDE analysis complete")

In [ ]:
# ----------------------------
# Cell 5 — DE analysis GSE176078
# Immune cells across breast cancer subtypes
# ----------------------------
adata2_immune = adata2[
    adata2.obs["cell_type"].isin(IMMUNE_TYPES)
].copy()
gc.collect()

immune_types_de = ["T cells", "CD8 T cells", "Macrophages", "NK cells"]
de_results2 = {}

for ct in immune_types_de:
    print(f"\nRunning DE for {ct} across subtypes...")
    
    adata_ct = adata2_immune[adata2_immune.obs["cell_type"] == ct].copy()
    counts = adata_ct.obs["subtype"].value_counts()
    print(f"  {counts.to_dict()}")
    
    if len(counts) < 2 or counts.min() < 20:
        print(f"  Skipping — not enough cells")
        continue
    
    sc.tl.rank_genes_groups(
        adata_ct,
        groupby="subtype",
        method="wilcoxon",
        key_added="de_subtype"
    )
    
    # Get results for each subtype
    for subtype in adata_ct.obs["subtype"].unique():
        de_df = sc.get.rank_genes_groups_df(
            adata_ct,
            group=subtype,
            key="de_subtype"
        )
        de_sig = de_df[
            (de_df["pvals_adj"] < 0.05) &
            (abs(de_df["logfoldchanges"]) > 0.5)
        ]
        print(f"  {subtype}: {len(de_sig)} significant DEGs")
        
        de_df.to_csv(
            RESULTS_DIR / f"GSE176078_DE_{ct.replace('/', '_').replace(' ', '_')}_{subtype}.csv",
            index=False
        )
    
    de_results2[ct] = adata_ct

print("\nDE analysis complete")

In [ ]:
# ----------------------------
# Cell 6 — Pathway enrichment (GSE114725)
# ----------------------------
print("Running pathway enrichment on DE results...")

for ct, de_df in de_results.items():
    print(f"\n{ct}:")
    
    # Get significant upregulated genes in TUMOR
    up_genes = de_df[
        (de_df["pvals_adj"] < 0.05) &
        (de_df["logfoldchanges"] > 0.5)
    ]["names"].tolist()
    
    if len(up_genes) < 10:
        print(f"  Too few significant genes ({len(up_genes)}) — skipping")
        continue
    
    print(f"  Upregulated genes: {len(up_genes)}")
    
    # Background = all genes in this cell type
    background = de_df["names"].tolist()
    
    try:
        enr = gp.enrichr(
            gene_list=up_genes,
            gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
            background=background,
            outdir=None,
            verbose=False
        )
        
        results = enr.results
        sig_pathways = results[results["Adjusted P-value"] < 0.05].copy()
        
        print(f"  Significant pathways: {len(sig_pathways)}")
        if len(sig_pathways) > 0:
            print(sig_pathways[["Gene_set", "Term", "Adjusted P-value", "Genes"]].head(5).to_string())
        
        results.to_csv(
            RESULTS_DIR / f"GSE114725_pathways_{ct.replace('/', '_').replace(' ', '_')}_tumor.csv",
            index=False
        )
        
    except Exception as e:
        print(f"  Enrichment failed: {e}")

print("\nPathway enrichment complete")

In [3]:
# ----------------------------
# Cell 7 — LIANA cell-cell communication (GSE114725)
# ----------------------------
print("Running LIANA on GSE114725...")

liana.mt.rank_aggregate(
    adata1,
    groupby="cell_type",
    resource_name="consensus",
    expr_prop=0.1,
    min_cells=10,
    verbose=True,
    use_raw=False
)

liana_res1 = adata1.uns["liana_res"]
sig1 = liana_res1[liana_res1["specificity_rank"] < 0.05].copy()

print(f"Total interactions: {len(liana_res1)}")
print(f"Significant: {len(sig1)}")

liana_res1.to_csv(RESULTS_DIR / "GSE114725_liana_results.csv", index=False)
sig1.to_csv(RESULTS_DIR / "GSE114725_liana_significant.csv", index=False)
print("Saved")

Using resource `consensus`.


Running LIANA on GSE114725...


Using `.X`!
Converting to sparse csr matrix!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
['TCONS_00029157'] contain `_`. Consider replacing those!
0.84 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 44662 samples and 122 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [01:41<00:00,  9.82it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt


Total interactions: 4644
Significant: 458
Saved


In [5]:
# Create immune subset for GSE176078
adata2_immune = adata2[
    adata2.obs["cell_type"].isin(IMMUNE_TYPES)
].copy()
gc.collect()

print(f"Immune cells: {adata2_immune.n_obs}")
print(adata2_immune.obs["cell_type"].value_counts())

Immune cells: 43820
cell_type
Naive/memory T cells    11590
CD8 T cells              9387
Macrophages              8705
T cells                  5989
B cells                  2791
Plasma cells             2583
NK cells                 2438
pDC                       315
Monocytes/DC               22
Name: count, dtype: int64


In [6]:
# ----------------------------
# Cell 8 — LIANA cell-cell communication (GSE176078)
# ----------------------------
print("Running LIANA on GSE176078 immune cells...")

# Use already subsetted immune object
liana.mt.rank_aggregate(
    adata2_immune,
    groupby="cell_type",
    resource_name="consensus",
    expr_prop=0.1,
    min_cells=10,
    verbose=True,
    use_raw=False
)

liana_res2 = adata2_immune.uns["liana_res"]
sig2 = liana_res2[liana_res2["specificity_rank"] < 0.05].copy()

print(f"Total interactions: {len(liana_res2)}")
print(f"Significant: {len(sig2)}")

liana_res2.to_csv(RESULTS_DIR / "GSE176078_liana_results.csv", index=False)
sig2.to_csv(RESULTS_DIR / "GSE176078_liana_significant.csv", index=False)
print("Saved")

Using resource `consensus`.


Running LIANA on GSE176078 immune cells...


Using `.X`!
Converting to sparse csr matrix!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
Converting `cell_type` to categorical!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.72 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 43820 samples and 259 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [03:04<00:00,  5.43it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt


Total interactions: 25434
Significant: 3086
Saved


In [8]:
# ----------------------------
# Cell 9 — Visualise LIANA results (heatmap approach)
# ----------------------------

def plot_liana_heatmap(liana_res, title, save_path, top_n=20):
    # Get top interactions
    top = liana_res[
        liana_res["specificity_rank"] < 0.05
    ].nsmallest(top_n, "specificity_rank").copy()
    
    # Create interaction label
    top["interaction"] = (
        top["ligand_complex"] + " → " + top["receptor_complex"]
    )
    top["pair"] = top["source"] + " → " + top["target"]
    
    # Pivot for heatmap
    pivot = top.pivot_table(
        index="interaction",
        columns="pair",
        values="specificity_rank",
        aggfunc="min"
    ).fillna(1)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    im = ax.imshow(
        pivot.values,
        cmap="RdYlBu_r",
        aspect="auto",
        vmin=0,
        vmax=0.05
    )
    
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    
    plt.colorbar(im, ax=ax, label="Specificity Rank")
    ax.set_title(title, fontsize=12, pad=15)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {save_path.name}")

# Load saved results
sig1 = pd.read_csv(RESULTS_DIR / "GSE114725_liana_significant.csv")
sig2 = pd.read_csv(RESULTS_DIR / "GSE176078_liana_significant.csv")

plot_liana_heatmap(
    sig1,
    "GSE114725 — Top Ligand-Receptor Interactions",
    FIGURE_DIR / "GSE114725_liana_heatmap.png"
)

plot_liana_heatmap(
    sig2,
    "GSE176078 — Top Ligand-Receptor Interactions (Immune cells)",
    FIGURE_DIR / "GSE176078_liana_heatmap.png"
)

Saved: GSE114725_liana_heatmap.png
Saved: GSE176078_liana_heatmap.png


In [ ]:
# ----------------------------
# Cell 10 — Visualise DE results
# ----------------------------

# Volcano plot function
def volcano_plot(de_df, title, save_path, lfc_thresh=0.5, pval_thresh=0.05):
    fig, ax = plt.subplots(figsize=(10, 8))
    
    de_df = de_df.copy()
    de_df["-log10_pval"] = -np.log10(de_df["pvals_adj"].clip(lower=1e-300))
    
    # Colour points
    de_df["colour"] = "grey"
    de_df.loc[
        (de_df["logfoldchanges"] > lfc_thresh) & (de_df["pvals_adj"] < pval_thresh),
        "colour"
    ] = "red"
    de_df.loc[
        (de_df["logfoldchanges"] < -lfc_thresh) & (de_df["pvals_adj"] < pval_thresh),
        "colour"
    ] = "blue"
    
    ax.scatter(
        de_df["logfoldchanges"],
        de_df["-log10_pval"],
        c=de_df["colour"],
        alpha=0.5,
        s=10
    )
    
    ax.axvline(x=lfc_thresh, color="black", linestyle="--", linewidth=0.8)
    ax.axvline(x=-lfc_thresh, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(y=-np.log10(pval_thresh), color="black", linestyle="--", linewidth=0.8)
    
    # Label top genes
    top_up = de_df[de_df["colour"] == "red"].nlargest(5, "logfoldchanges")
    top_down = de_df[de_df["colour"] == "blue"].nsmallest(5, "logfoldchanges")
    
    for _, row in pd.concat([top_up, top_down]).iterrows():
        ax.annotate(
            row["names"],
            (row["logfoldchanges"], row["-log10_pval"]),
            fontsize=7,
            ha="center"
        )
    
    ax.set_xlabel("Log2 Fold Change (Tumour vs Blood)")
    ax.set_ylabel("-log10 Adjusted P-value")
    ax.set_title(title)
    
    n_up = (de_df["colour"] == "red").sum()
    n_down = (de_df["colour"] == "blue").sum()
    ax.text(
        0.98, 0.98,
        f"Up: {n_up}\nDown: {n_down}",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=9
    )
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path.name}")

# Plot volcano for each cell type
for ct, de_df in de_results.items():
    volcano_plot(
        de_df,
        title=f"{ct} — Tumour vs Blood (GSE114725)",
        save_path=FIGURE_DIR / f"GSE114725_volcano_{ct.replace('/', '_').replace(' ', '_')}.png"
    )

print("Volcano plots saved")